# Klimatanalys och Avvikelser

Den här notebooken sammanställer och visualiserar klimatdata (temperatur och nederbörd) från SMHI:s lokala mätstationer och jämför dem med klimatnormaler (1991–2020). Syftet är att synliggöra hur de tillgängliga satellitårens väder förhöll sig till det historiska snittet.

### Importer
Importerar nödvändiga bibliotek, samt funktioner från `config.py` och `climate.py`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from config import (
    build_analysis_output_dirs, 
    load_and_prepare_scene_logs,
    select_all_clear_scenes,
    build_analysis_satellites,
    print_satellite_setup
)

from climate import (
    combine_stations, load_normal, load_normal_monthly,
    DATUM_COL, plot_climate_monthly_style_plot
)

### Globala inställningar och Matplotlib-tema
Sätter upp standardinställningar för plottarnas utseende som skriftstorlek, grid och bildupplösning.

In [ ]:
# ============================================
# Globalt tema
# ============================================
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.style.use("default") # matplotlibs standardstil 

mpl.rcParams.update({
    "font.size":          12,                            
    "axes.titlesize":     14,                            
    "axes.labelsize":     11,                            
    "xtick.labelsize":    12,                            
    "ytick.labelsize":    12,                            
    "legend.fontsize":    12,                            
    "axes.spines.top":    False, 
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.linestyle":     "--", 
    "grid.alpha":         0.4,
    "grid.color":         "#cccccc",
    "figure.dpi":         300,
    "figure.facecolor":   "white",
    "savefig.dpi":        300,
    "savefig.bbox":       "tight",
    "savefig.facecolor":  "white",
})

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================
area = "LF"  # OBS: här väljs studieområdet
sensor = "landsat"  # "landsat" eller "sentinel2"

satellites = build_analysis_satellites(area, sensor=sensor)

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_climate = analysis_output_dirs["climate_data"]
os.makedirs(output_dir_climate, exist_ok=True)

print_satellite_setup(area, satellites, sensor=sensor)

### Inläsning och sammanslagning av stationsdata
Här definieras sökvägar till uppmätt historisk klimatdata för olika väderstationer. Vissa perioder saknar data från en station, varför vi överlappar och kombinerar serier från näraliggande stationer för objekt- och referensområdet. Datan läses därefter in och sammanställs till enhetliga DataFrames.

In [ ]:
temp_data = {
    "Uppsala":        r"D:\Masteruppsats\Klimat\UppsalaAut_19850601_nu_temp.csv",
    "Svanberga":      r"D:\Masteruppsats\Klimat\Svanberga_19950801_nu_temp.csv",
    "Gavle":          r"D:\Masteruppsats\Klimat\Gavle_18581201_19950802_temp.csv",
    "Gavle_A":        r"D:\Masteruppsats\Klimat\GavleA_19950801_nu_temp.csv",
}

precip_data = {
    "Uppsala":        r"D:\Masteruppsats\Klimat\UppsalaAut_19850601_nu_ndb.csv",
    "Svanberga":      r"D:\Masteruppsats\Klimat\Svanberga_19950801_nu_ndb.csv",
    "Gavle":          r"D:\Masteruppsats\Klimat\Gavle_18851201_19950802_ndb.csv",
    "Gavle_A":        r"D:\Masteruppsats\Klimat\GavleA_19950901_nu_ndb.csv",
}

normal_period = {
    "temp_1991_2020":   r"D:\Masteruppsats\Klimat\Normal_temp_1991_2020.xlsx",
    "precip_1991_2020": r"D:\Masteruppsats\Klimat\Normal_nbd_1991_2020.xlsx"
}

# =============================================
# Stationskonfiguration
# =============================================
temp_stations_obj = [
    {"station": "Uppsala",         "file": temp_data["Uppsala"],        "start": "1985-05", "stop": "1988-08"},
    {"station": "Svanberga A",     "file": temp_data["Svanberga"],      "start": "1995-08", "stop": "2025-08"},
]
precip_stations_obj = [
    {"station": "Uppsala",     "file": precip_data["Uppsala"],   "start": "1985-05", "stop": "1995-07"},
    {"station": "Svanberga A", "file": precip_data["Svanberga"], "start": "1995-08", "stop": "2025-08"},
]
temp_stations_ref = [
    {"station": "Gävle",   "file": temp_data["Gavle"],   "start": "1984-05", "stop": "1995-07"},
    {"station": "Gävle A", "file": temp_data["Gavle_A"], "start": "1995-08", "stop": "2025-08"},
]
precip_stations_ref = [
    {"station": "Gävle",   "file": precip_data["Gavle"],   "start": "1984-05", "stop": "1995-07"},
    {"station": "Gävle A", "file": precip_data["Gavle_A"], "start": "1995-08", "stop": "2025-08"},
]

# =============================================
# Läs in och kombinera stationer
# =============================================
df_temp_obj   = combine_stations(temp_stations_obj,   "Lufttemperatur").rename(columns={"Lufttemperatur": "Air temperature"})
df_precip_obj = combine_stations(precip_stations_obj, "Nederbördsmängd").rename(columns={"Nederbördsmängd": "Precipitation"})
df_temp_ref   = combine_stations(temp_stations_ref,   "Lufttemperatur").rename(columns={"Lufttemperatur": "Air temperature"})
df_precip_ref = combine_stations(precip_stations_ref, "Nederbördsmängd").rename(columns={"Nederbördsmängd": "Precipitation"})

### Klimatnormaler (1991–2020)
Officiella referensvärden ("normaler") för den nuvarande normalperioden hämtas och lagras. Genom att jämföra årliga mätningar med detta snitt kan vi visualisera månadsvisa avvikelser ("anomalier"). Normalvärdena loggas sedan nere i utskriften.

In [ ]:
temp_normal_obj_monthly   = load_normal_monthly(normal_period["temp_1991_2020"],   "Svanberga A")
precip_normal_obj_monthly = load_normal_monthly(normal_period["precip_1991_2020"], "Svanberga A")
temp_normal_ref_monthly   = load_normal_monthly(normal_period["temp_1991_2020"],   "Gävle A")
precip_normal_ref_monthly = load_normal_monthly(normal_period["precip_1991_2020"], "Gävle A")


# =============================================
# Normalvärden (1991-2020)
# =============================================
temp_normal_obj   = load_normal(normal_period["temp_1991_2020"],   "Svanberga A")
precip_normal_obj = load_normal(normal_period["precip_1991_2020"], "Svanberga A")
temp_normal_ref   = load_normal(normal_period["temp_1991_2020"],   "Gävle A")
precip_normal_ref = load_normal(normal_period["precip_1991_2020"], "Gävle A")

print(f"Normaltemp   GRM/GM/LF/GS (Svanberga A): {temp_normal_obj:.2f}°C")
print(f"Normalprecip GRM/GM/LF/GS (Svanberga A): {precip_normal_obj:.1f} mm")
print(f"Normaltemp   TAM/AM       (Gävle A):      {temp_normal_ref:.2f}°C")
print(f"Normalprecip TAM/AM       (Gävle A):      {precip_normal_ref:.1f} mm")

### Tillgängliga satellitår
Scenloggarna filtreras för att identifiera exakt vilka maj–augusti-månader som bistod med godkända, molnfria satellitbilder. Detta underlättar ifall man vill koppla väderdata rent konkret till de år man faktiskt genomför bildanalys på.

In [ ]:
# Anpassa efter vilket område du kör för
log = load_and_prepare_scene_logs(satellites, area)
log = log.sort_values("date").reset_index(drop=True)

all_clear  = select_all_clear_scenes(log, month_start=5, month_end=8)
scene_years = sorted(all_clear["date"].dt.year.unique())

### Visualisering av klimatanomalier
Sist genereras de slutgiltiga graferna över avvikelser i både temperatur och nederbörd. Värden aggregeras (medel för temperatur, summa/procentuell förändring för nederbörd) och ritas upp för objekt- respektive referensstationerna. Resultaten sparas sedan som PNG i output-mappen.

In [ ]:
plot_climate_monthly_style_plot(
    df             = df_temp_obj,
    value_col      = "Air temperature",
    normal_monthly = temp_normal_obj_monthly,
    ylabel         = "Deviation (°C)",
    title          = "Temperature anomaly May–Aug relative to normal period (1991–2020) – Svanberga A",
    output_path    = os.path.join(output_dir_climate, "GRM_temp_anomaly_monthly.png"),
    agg            = "mean",
    mode           = "diff",
    #scene_years    = scene_years,
)

plot_climate_monthly_style_plot(
    df             = df_precip_obj,
    value_col      = "Precipitation",
    normal_monthly = precip_normal_obj_monthly,
    ylabel         = "Deviation (%)",
    title          = "Precipitation anomaly May–Aug relative to normal period (1991–2020) – Svanberga A",
    output_path    = os.path.join(output_dir_climate, "GRM_precip_anomaly_monthly.png"),
    agg            = "sum",
    mode           = "percent",
    #scene_years    = scene_years,
)

plot_climate_monthly_style_plot(
    df             = df_temp_ref,
    value_col      = "Air temperature",
    normal_monthly = temp_normal_ref_monthly,
    ylabel         = "Deviation (°C)",
    title          = "Temperature anomaly May–Aug relative to normal period (1991–2020) – Gävle A",
    output_path    = os.path.join(output_dir_climate, "TAM_temp_anomaly_monthly.png"),
    agg            = "mean",
    mode           = "diff",
    #scene_years    = scene_years,
)

plot_climate_monthly_style_plot(
    df             = df_precip_ref,
    value_col      = "Precipitation",
    normal_monthly = precip_normal_ref_monthly,
    ylabel         = "Deviation (%)",
    title          = "Precipitation anomaly May–Aug relative to normal period (1991–2020) – Gävle A",
    output_path    = os.path.join(output_dir_climate, "TAM_precip_anomaly_monthly.png"),
    agg            = "sum",
    mode           = "percent",
    #scene_years    = scene_years,
)
print(f"Climate plots saved to {output_dir_climate}")